In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('dataset.csv')
df = df.dropna()

# Dữ liệu Input
X = df[['avg_temp', 'avg_humi']].values

y = df['anomaly'].values

# Chia train/test 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Chuẩn hóa dữ liệu (Z-score Normalization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"const float TEMP_MEAN = {scaler.mean_[0]:.2f}f; const float TEMP_STD = {scaler.scale_[0]:.2f}f;")
print(f"const float HUMI_MEAN = {scaler.mean_[1]:.2f}f; const float HUMI_STD = {scaler.scale_[1]:.2f}f;")

print("Kích thước tập Train:", X_train_scaled.shape)
print("Kích thước tập Test:", X_test_scaled.shape)
print("\nPhân phối nhãn (0 = Bình thường, 1 = Bất thường):")
print(df['anomaly'].value_counts())

# Khởi tạo và huấn luyện mô hình
model = Sequential([
    Dense(16, activation='relu', input_shape=(2,)), # Nhận 2 đầu vào (Nhiệt độ, Độ ẩm)
    Dropout(0.2),                                   # Chống Overfitting
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("\nĐang huấn luyện mô hình...")
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop]
)

# Đánh giá mô hình
loss, accuracy = model.evaluate(X_test_scaled, y_test)
print(f"\nĐộ chính xác trên tập Test: {accuracy * 100:.2f}%\n")

# Chuyển đổi sang TFLite và xuất ra C++
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

def hex_to_c_array(hex_data, var_name):
    c_str = '#ifndef MY_DHT_ANOMALY_MODEL_H\n#define MY_DHT_ANOMALY_MODEL_H\n\n'
    c_str += f'const unsigned char {var_name}[] = {{\n'

    hex_array = [f'0x{val:02x}' for val in hex_data]

    for i in range(0, len(hex_array), 12):
        c_str += '  ' + ', '.join(hex_array[i:i+12]) + ',\n'

    c_str = c_str.rstrip(',\n') + '\n};\n\n'
    c_str += f'const unsigned int {var_name}_len = {len(hex_data)};\n\n'
    c_str += '#endif // MY_DHT_ANOMALY_MODEL_H'
    return c_str

c_code = hex_to_c_array(tflite_model, "dht_anomaly_model_tflite")

with open('my_dht_anomaly_model.h', 'w') as f:
    f.write(c_code)

print("Tạo xong file 'my_dht_anomaly_model.h'!")
print(f"Kích thước file mô hình C++: {len(tflite_model)} bytes")


const float TEMP_MEAN = 26.30f; const float TEMP_STD = 5.95f;
const float HUMI_MEAN = 62.80f; const float HUMI_STD = 17.26f;
Kích thước tập Train: (800, 2)
Kích thước tập Test: (200, 2)

Phân phối nhãn (0 = Bình thường, 1 = Bất thường):
anomaly
1    500
0    500
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Đang huấn luyện mô hình...
Epoch 1/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.6281 - loss: 0.6822 - val_accuracy: 0.6875 - val_loss: 0.6583
Epoch 2/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7469 - loss: 0.6244 - val_accuracy: 0.7750 - val_loss: 0.5994
Epoch 3/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8000 - loss: 0.5619 - val_accuracy: 0.8000 - val_loss: 0.5185
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8516 - loss: 0.4800 - val_accuracy: 0.8313 - val_loss: 0.4274
Epoch 5/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.8922 - loss: 0.3994 - val_accuracy: 0.9250 - val_loss: 0.3435
Epoch 6/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9250 - loss: 0.3240 - val_accuracy: 0.9625 - val_loss: 0.2771
Epoch 7/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9328 - loss: 0.2755 - val_accuracy: 0.9375 - val_loss: 0.2244
Epoch 8/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9219 - los